# Final Submission Notebook

Notebook ini berisi pipeline final untuk membuat file submission Kaggle pada prediksi `property_organic_content`.

Output akhir:

- `submission.csv`

Model final yang dipakai adalah focused blend terpilih dengan public RMSE terbaik saat ini: **12.04499**.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SUBMISSION_DIR = ROOT / "submissions"
TARGET = "property_organic_content"
FINAL_SOURCE = SUBMISSION_DIR / "focused_anchor_ridge40.csv"
FINAL_OUTPUT = ROOT / "submission.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.5f}")

## 1. Load Data

Data train, test, dan sample submission dibaca untuk memastikan struktur file final sesuai format kompetisi.

In [2]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"Train shape: {train.shape}")
print(f"Test shape : {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

assert TARGET in train.columns
assert TARGET not in test.columns
assert list(sample_submission.columns) == ["sample_id", TARGET]
assert sample_submission["sample_id"].equals(test["sample_id"])

Train shape: (11210, 52)
Test shape : (2670, 51)
Sample submission shape: (2670, 2)


## 2. Load Final Prediction

File final prediction dipakai sebagai sumber submission. File ini adalah kandidat terbaik yang sudah divalidasi pada Kaggle public leaderboard.

In [3]:
final_submission = pd.read_csv(FINAL_SOURCE)

assert list(final_submission.columns) == list(sample_submission.columns)
assert len(final_submission) == len(sample_submission)
assert final_submission["sample_id"].equals(sample_submission["sample_id"])
assert final_submission[TARGET].notna().all()
assert np.isfinite(final_submission[TARGET]).all()

print(f"Final source loaded: {FINAL_SOURCE}")
display(final_submission.head())

Final source loaded: C:\Users\nicho\Documents\Projects\compfest\submissions\focused_anchor_ridge40.csv


,sample_id,property_organic_content
0,test_00001,23.00338
1,test_00002,13.53828
2,test_00003,27.98391
3,test_00004,23.36962
4,test_00005,29.47209


## 3. Sanity Check Prediction Distribution

Distribusi prediksi dicek untuk memastikan tidak ada nilai kosong, tak hingga, atau pola ekstrem yang tidak wajar.

In [4]:
prediction_summary = final_submission[TARGET].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).to_frame("final_prediction")

display(prediction_summary)

,final_prediction
count,"2,670.00000"
mean,34.57482
std,20.70008
min,6.01630
1%,10.92649
5%,13.56131
50%,28.15349
95%,76.36152
99%,108.57104
max,159.15685


## 4. Write Submission File

File `submission.csv` ditulis ulang dari final prediction dan langsung divalidasi terhadap `sample_submission.csv`.

In [5]:
final_submission.to_csv(FINAL_OUTPUT, index=False)
check = pd.read_csv(FINAL_OUTPUT)

assert list(check.columns) == list(sample_submission.columns)
assert len(check) == len(sample_submission)
assert check["sample_id"].equals(sample_submission["sample_id"])
assert check[TARGET].notna().all()
assert np.isfinite(check[TARGET]).all()

print(f"Submission file written: {FINAL_OUTPUT}")
print("Validation passed.")

Submission file written: C:\Users\nicho\Documents\Projects\compfest\submission.csv
Validation passed.


## 5. Final Status

`submission.csv` sudah siap diupload ke Kaggle.

Ringkasan:

- target: `property_organic_content`
- jumlah baris submission: 2,670
- public RMSE terbaik saat ini: **12.04499**
- file output: `submission.csv`